# IDPFold2 Monomer Preview (Colab)

This notebook runs IDPFold2 for an example, a custom sequence, or an uploaded CSV, then downloads the generated `.pdb` files.

Use **Runtime > Change runtime type > GPU** for practical inference speed.

In [ ]:
#@title <b>1. Install Conda runtime</b> { display-mode: "form" }
#@markdown Run this cell first. It installs `condacolab` and restarts the Colab runtime when finished.
import subprocess
subprocess.run( 'pip install -q condacolab'.split())

import condacolab
condacolab.install()

In [ ]:
#@title <b>2. Install IDPFold2 environment</b> { display-mode: "form" }
#@markdown Run this cell after the runtime restart from step 1. It clones IDPFold2, installs dependencies, and verifies the runtime imports.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Junjie-Zhu/IDPFold2'
REPO_DIR = Path('/content/IDPFold-multimer')


def run(command):
    print('$', ' '.join(str(part) for part in command))
    subprocess.run(command, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

PIP_DEPS = [
    'torch==2.4.1',
    'torch-geometric==2.6.1',
    'einops==0.6',
    'dm-tree==0.1.8',
    'loguru==0.7.2',
    'hydra-core==1.3.1',
    'pandas',
    'scipy',
    'numpy==1.26.0',
    'biotite==0.41.0',
    'biopandas==0.5.1',
    'wget==3.2',
    'tqdm==4.66.4',
    'cpdb-protein',
    'biopython',
    'rootutils',
    'pytest',
    'gdown'
]
run([sys.executable, '-m', 'pip', 'install', *PIP_DEPS])

# Runtime extras needed by this Colab notebook but not listed in environment.yaml.
NOTEBOOK_DEPS = ['fair-esm', 'scipy', 'ipywidgets']
run([sys.executable, '-m', 'pip', 'install', *NOTEBOOK_DEPS])
run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--no-deps'])

if not (REPO_DIR / 'src' / 'inference.py').exists():
    raise FileNotFoundError(f'IDPFold2 source tree was not found under {REPO_DIR}')

os.chdir(REPO_DIR)
repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import numpy as np
import pandas as pd
import scipy
import torch
import esm
import hydra
import src.inference

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Environment is ready:', REPO_DIR)

In [ ]:
#@title <b>3. Download model checkpoint</b> { display-mode: "form" }
#@markdown This step downloads the IDPFold2 checkpoint from the configured Google Drive file and validates it before inference.
import subprocess
import sys
import torch
import wget

CHECKPOINT_SOURCE = 'google_drive'
GDRIVE_FILE_ID = '1HX6b24UUhsMl8NQRxqZmcg65MjTGoH6y'
CHECKPOINT_URL = 'https://zenodo.org/records/18239596/files/IDPFold2_ema_0.999_260114.pth?download=1'
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CHECKPOINT_DIR / 'IDPFold2_ema_0.999_260114.pth'
TMP_CKPT_PATH = CKPT_PATH.with_suffix('.tmp')
MIN_CHECKPOINT_BYTES = 1024 * 1024


def checkpoint_is_valid(path):
    if not path.exists():
        return False
    size = path.stat().st_size
    print(f'Checkpoint size: {size / 1024**2:.1f} MiB')
    if size < MIN_CHECKPOINT_BYTES:
        print('Checkpoint is too small to be a valid PyTorch checkpoint.')
        return False
    try:
        checkpoint = torch.load(path, map_location='cpu')
    except Exception as exc:
        print(f'Checkpoint validation failed: {type(exc).__name__}: {exc}')
        return False
    del checkpoint
    return True


def ensure_gdown():
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        import gdown
    return gdown


def download_checkpoint():
    TMP_CKPT_PATH.unlink(missing_ok=True)
    print('Downloading checkpoint...')
    if CHECKPOINT_SOURCE == 'google_drive':
        gdown = ensure_gdown()
        output = gdown.download(id=GDRIVE_FILE_ID, output=str(TMP_CKPT_PATH), quiet=False)
        if output is None:
            raise RuntimeError(
                'Google Drive download failed. Make sure the file sharing setting is "Anyone with the link".'
            )
    elif CHECKPOINT_SOURCE == 'url':
        wget.download(CHECKPOINT_URL, str(TMP_CKPT_PATH))
        print()
    else:
        raise ValueError(f'Unsupported CHECKPOINT_SOURCE: {CHECKPOINT_SOURCE}')
    TMP_CKPT_PATH.replace(CKPT_PATH)


if not checkpoint_is_valid(CKPT_PATH):
    CKPT_PATH.unlink(missing_ok=True)
    download_checkpoint()

if not checkpoint_is_valid(CKPT_PATH):
    CKPT_PATH.unlink(missing_ok=True)
    raise RuntimeError(f'Checkpoint download failed validation: {CKPT_PATH}')

print('Checkpoint:', CKPT_PATH)

In [ ]:
#@title <b>4. Prepare input sequences</b> { display-mode: "form" }
#@markdown Choose one input mode. The notebook writes the selected input to `/content/input_monomer.csv` for IDPFold2.
import re
import pandas as pd
from google.colab import files

WORK_DIR = REPO_DIR
INPUT_CSV = Path('/content/input_monomer.csv')
ALLOWED_AA = set('ACDEFGHIKLMNPQRSTVWYX')

#@markdown ### Input source
INPUT_MODE = 'example'  # @param ['example', 'custom_sequence', 'csv_upload']

#@markdown ### Example input
#@markdown Used only when `INPUT_MODE` is `example`. Enter `all` to run every row in `data/monomer_example.csv`.
EXAMPLE_TEST_CASE = 'THB_C2'  # @param {type:'string'}

#@markdown ### Custom sequence name
#@markdown Used only when `INPUT_MODE` is `custom_sequence`. This becomes the output file stem.
CUSTOM_TEST_CASE = 'custom_demo'  # @param {type:'string'}

#@markdown ### Custom sequence
#@markdown Used only when `INPUT_MODE` is `custom_sequence`. Paste amino acid letters only; whitespace is removed automatically.
CUSTOM_SEQUENCE = 'GPGSEDVWEILRQAPPSEYERIAFQYGVTDLRGMLKRLKGMRRDEKKSTAFQKKLEPAYQVSKGHKIRLTVELADHDAEVKWLKNGQEIQMSGSKYIFESIGAKRTLTISQCSLADDAAYQCVVGGEKCSTELFVKE'  # @param {type:'string'}

#@markdown ### CSV upload format
#@markdown Used only when `INPUT_MODE` is `csv_upload`. Upload a CSV with exactly these required columns:
#@markdown `test_case,sequence`
#@markdown Example row: `custom_demo,ACDEFGHIKLMNPQRSTVWY`


def sanitize_name(name):
    name = re.sub(r'[^A-Za-z0-9_.-]+', '_', name.strip())
    return name or 'custom_demo'


def normalize_sequence(sequence):
    return re.sub(r'\s+', '', sequence).upper()


def validate_inputs(df):
    required = {'test_case', 'sequence'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Input CSV is missing columns: {sorted(missing)}')
    df = df.loc[:, ['test_case', 'sequence']].copy()
    df['test_case'] = df['test_case'].map(lambda value: sanitize_name(str(value)))
    df['sequence'] = df['sequence'].map(lambda value: normalize_sequence(str(value)))
    if df['test_case'].duplicated().any():
        raise ValueError('test_case values must be unique.')
    for row in df.itertuples(index=False):
        if not row.sequence:
            raise ValueError(f'{row.test_case} has an empty sequence.')
        invalid = sorted(set(row.sequence) - ALLOWED_AA)
        if invalid:
            raise ValueError(f'{row.test_case} contains unsupported residue codes: {invalid}')
    return df


if INPUT_MODE == 'example':
    example_csv = WORK_DIR / 'data' / 'monomer_example.csv'
    input_df = pd.read_csv(example_csv)
    if EXAMPLE_TEST_CASE.strip().lower() != 'all':
        input_df = input_df[input_df['test_case'] == EXAMPLE_TEST_CASE.strip()]
        if input_df.empty:
            raise ValueError(f'Example test case not found: {EXAMPLE_TEST_CASE}')
elif INPUT_MODE == 'custom_sequence':
    input_df = pd.DataFrame([{'test_case': CUSTOM_TEST_CASE, 'sequence': CUSTOM_SEQUENCE}])
elif INPUT_MODE == 'csv_upload':
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No CSV file uploaded.')
    uploaded_name = next(iter(uploaded))
    input_df = pd.read_csv(uploaded_name)
else:
    raise ValueError(f'Unsupported INPUT_MODE: {INPUT_MODE}')

input_df = validate_inputs(input_df)
input_df.to_csv(INPUT_CSV, index=False)
EXPECTED_TEST_CASES = input_df['test_case'].tolist()

print('Input CSV:', INPUT_CSV)
print('Test cases:', ', '.join(EXPECTED_TEST_CASES))
display(input_df)

In [ ]:
#@title <b>5. Run IDPFold2 inference</b> { display-mode: "form" }
#@markdown Configure the run name and sampling settings, then execute IDPFold2.
import os
import shlex

#@markdown ### Output prefix
PREFIX = 'COLAB_MONOMER'  # @param {type:'string'}

#@markdown ### Sampling settings
NSAMPLES = 4              # @param {type:'integer'}
MAX_BATCH_LENGTH = 3500   # @param {type:'integer'}
NUM_WORKERS = 0           # @param {type:'integer'}

PLM_EMB_DIR = Path('/content/embeddings')
LOGGING_DIR = Path('/content/outputs')
PLM_EMB_DIR.mkdir(parents=True, exist_ok=True)
LOGGING_DIR.mkdir(parents=True, exist_ok=True)

# Run the script file directly so Hydra resolves ../configs from the repo checkout.
entrypoint = [sys.executable, str(WORK_DIR / 'src' / 'inference.py')]
cmd = entrypoint + [
    f'prefix={PREFIX}',
    f'ckpt_dir={CKPT_PATH}',
    f'plm_emb_dir={PLM_EMB_DIR}',
    f'csv_dir={INPUT_CSV}',
    f'nsamples={NSAMPLES}',
    f'max_batch_length={MAX_BATCH_LENGTH}',
    f'num_workers={NUM_WORKERS}',
    f'logging_dir={LOGGING_DIR}',
]

command_text = ' '.join(shlex.quote(str(x)) for x in cmd)
env = os.environ.copy()
env['HYDRA_FULL_ERROR'] = '1'
env['PYTHONFAULTHANDLER'] = '1'

print('Running:', command_text)
print('Working directory:', WORK_DIR)
result = subprocess.run(
    cmd,
    check=False,
    cwd=WORK_DIR,
    env=env,
    text=True,
    capture_output=True,
)

if result.stdout:
    print('\n=== STDOUT ===')
    print(result.stdout, end='' if result.stdout.endswith('\n') else '\n')
if result.stderr:
    print('\n=== STDERR ===')
    print(result.stderr, end='' if result.stderr.endswith('\n') else '\n')
if result.returncode != 0:
    print('\n=== COMMAND FAILED ===')
    print('Exit code:', result.returncode)
    print('Command:', command_text)
    print('Working directory:', WORK_DIR)
    raise RuntimeError(f'IDPFold2 inference failed with exit code {result.returncode}. See STDOUT/STDERR above.')

run_dirs = sorted(
    [path for path in LOGGING_DIR.glob(f'{PREFIX}_INF_*') if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
)
if not run_dirs:
    raise FileNotFoundError(f'No output directory was created under {LOGGING_DIR}')

LATEST_RUN_DIR = run_dirs[-1]
PDB_FILES = {path.stem: path for path in sorted((LATEST_RUN_DIR / 'samples').glob('*.pdb'))}
if not PDB_FILES:
    raise FileNotFoundError(f'No PDB files were generated in {LATEST_RUN_DIR / "samples"}')

print('Output directory:', LATEST_RUN_DIR)
print('Generated PDB files:')
for name, path in PDB_FILES.items():
    print(f'  {name}: {path}')

In [ ]:
#@title <b>6. Download generated PDB files</b> { display-mode: "form" }
#@markdown Download one selected PDB file or zip all generated PDB files from the latest run.
import zipfile
from google.colab import files

#@markdown ### Download option
DOWNLOAD_MODE = 'selected'  # @param ['selected', 'all']

#@markdown ### Selected structure
#@markdown Leave empty to download the first generated test case. Used only when `DOWNLOAD_MODE` is `selected`.
DOWNLOAD_TEST_CASE = ''  # @param {type:'string'}

if DOWNLOAD_MODE == 'all':
    ZIP_PATH = LATEST_RUN_DIR / 'idpfold2_colab_pdbs.zip'
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in PDB_FILES.values():
            archive.write(path, arcname=path.name)
    print(f'Download all PDB files: {ZIP_PATH}')
    files.download(str(ZIP_PATH))
elif DOWNLOAD_MODE == 'selected':
    selected_name = DOWNLOAD_TEST_CASE.strip() or next(iter(PDB_FILES))
    if selected_name not in PDB_FILES:
        raise ValueError(f'Unknown DOWNLOAD_TEST_CASE {selected_name!r}. Available: {list(PDB_FILES)}')
    PDB_PATH = PDB_FILES[selected_name]
    print(f'Download selected PDB: {PDB_PATH}')
    files.download(str(PDB_PATH))
else:
    raise ValueError(f'Unsupported DOWNLOAD_MODE: {DOWNLOAD_MODE}')